# AnimeLoom — RunPod Text-to-Anime

Generate studio-quality anime video from text on RunPod GPUs.

| Stage | What | Purpose |
|-------|------|---------|
| **Story Decomposer** | Gemini Flash (free) or rule-based | Text → shot script |
| **SDXL + LoRA** | Keyframe generation | Character identity via trained LoRA |
| **AnimateDiff T2V** | Text-to-video + IP-Adapter | Real motion with character conditioning |
| **Motion Trim** | Remove static tails | Cleaner clip endings |
| **RIFE** | Temporal upscale 8→24fps | Smooth animation |
| **Real-ESRGAN** | Spatial sharpening | Crisp anime lines |
| **Face Restore** | Anime face enhancement | Sharper faces |
| **Color Grade** | Anime palette | Vibrant studio colors |
| **Assembly** | Cross-dissolve stitch | Smooth scene transitions |

In [ ]:
#@title 1. Setup Environment
#@markdown Run this cell once after pod starts. Installs pinned dependencies.

import subprocess, sys, os
from pathlib import Path

# Clone repo if needed
os.chdir("/workspace")
if not Path("/workspace/AnimeLoom").exists():
    subprocess.run(["git", "clone", "https://github.com/JoelJohnsonThomas/AnimeLoom.git"], check=True)
os.chdir("/workspace/AnimeLoom")
sys.path.insert(0, "/workspace/AnimeLoom")

# Pinned deps — compatible with base image torch 2.4.1+cu124
deps = [
    "diffusers==0.30.3", "transformers==4.44.2", "accelerate==0.33.0", "numpy<2.0",
    "safetensors", "peft", "sentencepiece", "protobuf",
    "opencv-python-headless", "pillow", "scikit-image", "scikit-learn",
    "controlnet-aux", "einops", "omegaconf",
    "realesrgan", "basicsr==1.4.2", "gfpgan", "facexlib",
    "google-genai", "google-generativeai",
    "huggingface_hub",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + deps)

# Fix total_mem → total_memory for torch 2.4
wan_path = Path("agents/animator/wan_wrapper.py")
if wan_path.exists():
    code = wan_path.read_text()
    if ".total_mem" in code and ".total_memory" not in code:
        wan_path.write_text(code.replace(".total_mem", ".total_memory"))
        print("Fixed total_mem → total_memory")

# Set warehouse
os.environ["AI_CACHE_ROOT"] = "/workspace/warehouse"
for d in ["models", "lora", "outputs", "checkpoints", "references"]:
    Path(f"/workspace/warehouse/{d}").mkdir(parents=True, exist_ok=True)

# torchvision compatibility shim for basicsr
import torchvision, types
if not hasattr(torchvision.transforms, 'functional_tensor'):
    import torchvision.transforms.functional as F
    m = types.ModuleType('torchvision.transforms.functional_tensor')
    for a in ['rgb_to_grayscale', 'normalize', 'resize', 'pad']:
        if hasattr(F, a): setattr(m, a, getattr(F, a))
    sys.modules['torchvision.transforms.functional_tensor'] = m

import torch
print(f"torch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB)")
print(f"Warehouse: {os.environ['AI_CACHE_ROOT']}")
print("Setup complete!")

In [ ]:
#@title 2. Download Character LoRA from HuggingFace
#@markdown Downloads pre-trained character LoRA. Skip if you'll train your own.

CHARACTER_REPO = "joelthomas77/animeloom-sakura-haruno"  #@param {type:"string"}
CHARACTER_NAME = "sakura_haruno"  #@param {type:"string"}

from huggingface_hub import snapshot_download
import shutil
from pathlib import Path

WAREHOUSE = Path("/workspace/warehouse")
hf_dir = WAREHOUSE / "lora" / "_hf_download"

print(f"Downloading {CHARACTER_REPO}...")
snapshot_download(CHARACTER_REPO, local_dir=str(hf_dir))

# Place SDXL and SD 1.5 LoRAs in correct directories
for src, suffix in [("sdxl", ""), ("sd15", "_sd15")]:
    src_dir = hf_dir / src
    if src_dir.exists():
        dst_dir = WAREHOUSE / "lora" / f"{CHARACTER_NAME}{suffix}"
        dst_dir.mkdir(parents=True, exist_ok=True)
        for f in src_dir.iterdir():
            shutil.copy2(f, dst_dir / f.name)
        print(f"  {src} → {dst_dir} ({sum(1 for _ in dst_dir.iterdir())} files)")

# Verify
sdxl_ok = (WAREHOUSE / "lora" / CHARACTER_NAME / "adapter_model.safetensors").exists()
sd15_ok = (WAREHOUSE / "lora" / f"{CHARACTER_NAME}_sd15" / "adapter_model.safetensors").exists()
print(f"\nSDXL LoRA: {'OK' if sdxl_ok else 'MISSING'}")
print(f"SD 1.5 LoRA (AnimateDiff): {'OK' if sd15_ok else 'MISSING'}")

In [ ]:
#@title 3. Text-to-Anime Generation
#@markdown Type a story and get anime video with character consistency.

#@markdown ---
#@markdown **Your Story**
STORY_TEXT = "Sakura Haruno walks through a cherry blossom forest at sunset. Petals fall around her as she stops at a wooden bridge and looks at the river below. The wind gently moves her pink hair."  #@param {type:"string"}

#@markdown ---
#@markdown **Character** (must match LoRA from Step 2, or leave empty)
CHARACTER_NAME = "sakura_haruno"  #@param {type:"string"}

#@markdown ---
#@markdown **Gemini API Key** (optional — get free at aistudio.google.com/apikey)
GEMINI_API_KEY = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Video Settings**
NUM_FRAMES = 24  #@param {type:"slider", min:16, max:32, step:4}
ANIM_STEPS = 30  #@param {type:"slider", min:15, max:50, step:5}
ANIM_GUIDANCE = 8.0  #@param {type:"slider", min:3.0, max:15.0, step:0.5}
IP_ADAPTER_SCALE = 0.6  #@param {type:"slider", min:0.0, max:1.0, step:0.1}
FPS = 8  #@param {type:"slider", min:8, max:16, step:4}
TARGET_FPS = 24  #@param {type:"slider", min:8, max:30, step:4}

#@markdown ---
#@markdown **Post-Processing**
FACE_RESTORE = True  #@param {type:"boolean"}
SPATIAL_UPSCALE = True  #@param {type:"boolean"}
COLOR_GRADE = True  #@param {type:"boolean"}
COLOR_PALETTE = "warm"  #@param ["warm", "cool", "vibrant", "muted"]

#@markdown ---
#@markdown **SDXL Keyframe Settings**
IMAGE_WIDTH = 768  #@param {type:"slider", min:512, max:1024, step:128}
IMAGE_HEIGHT = 1152  #@param {type:"slider", min:512, max:1152, step:128}
SDXL_STEPS = 25  #@param {type:"slider", min:10, max:50, step:5}
SDXL_GUIDANCE = 7.5  #@param {type:"slider", min:1.0, max:15.0, step:0.5}
LORA_SCALE = 1.0  #@param {type:"slider", min:0.5, max:2.0, step:0.1}
NEGATIVE_PROMPT = "blurry, low quality, deformed, extra fingers, worst quality, ugly, disfigured, bad anatomy, bad hands, 3d render, cgi, photorealistic, live action"  #@param {type:"string"}

import os, gc, sys, time, torch, json
import numpy as np
from pathlib import Path
from PIL import Image
import cv2

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# ================================================================
# Phase 1: Story Decomposition
# ================================================================
print("=" * 60)
print("PHASE 1: Story Decomposition")
print("=" * 60)

from agents.story.decomposer import StoryDecomposer
decomposer = StoryDecomposer(gemini_api_key=GEMINI_API_KEY or None, character_name=CHARACTER_NAME or None)
script = decomposer.decompose(STORY_TEXT)
shots = decomposer.decompose_to_shots(STORY_TEXT)
print(f"\nGenerated {len(shots)} shots:")
for i, s in enumerate(shots):
    print(f"  Shot {i+1}: {s['description'][:80]}...")

# ================================================================
# Phase 2: SDXL + LoRA Keyframes
# ================================================================
print("\n" + "=" * 60)
print("PHASE 2: SDXL + LoRA Keyframes")
print("=" * 60)

gc.collect(); torch.cuda.empty_cache()

char_id = CHARACTER_NAME.lower().replace(" ", "_") if CHARACTER_NAME else None
lora_dir = WAREHOUSE / "lora" / char_id if char_id else None
has_lora = lora_dir and (lora_dir / "metadata.json").exists() if lora_dir else False

from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

if has_lora:
    from peft import PeftModel
    meta = json.loads((lora_dir / "metadata.json").read_text())
    base_model = meta.get("base_model", "cagliostrolab/animagine-xl-3.1")
    # Fix local cache paths from Colab training
    if base_model.startswith("/") or base_model.startswith("C:"):
        base_model = "cagliostrolab/animagine-xl-3.1"
    print(f"Loading {base_model} + LoRA for '{CHARACTER_NAME}'...")
else:
    base_model = "cagliostrolab/animagine-xl-3.1"
    print(f"Loading {base_model} (no character LoRA)...")

pipe = StableDiffusionXLPipeline.from_pretrained(
    base_model, torch_dtype=torch.float16,
    cache_dir=str(WAREHOUSE / "models"),
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

if has_lora and (lora_dir / "adapter_model.safetensors").exists():
    pipe.unet = PeftModel.from_pretrained(pipe.unet, str(lora_dir))
    pipe.unet.eval()
    if LORA_SCALE != 1.0:
        for module in pipe.unet.modules():
            if hasattr(module, "scaling"):
                for key in module.scaling:
                    module.scaling[key] = LORA_SCALE
    print(f"  LoRA loaded (scale={LORA_SCALE})")

pipe.to("cuda")
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
try: pipe.enable_xformers_memory_efficient_attention()
except: pass

keyframes = []
for i, shot in enumerate(shots):
    desc = shot["description"]
    if CHARACTER_NAME and CHARACTER_NAME.lower() not in desc.lower():
        desc = f"{CHARACTER_NAME}, {desc}"
    prompt = f"{desc}, anime style, anime screencap, studio quality, sharp lineart, vibrant colors, masterpiece, best quality"
    gen = torch.Generator("cuda").manual_seed(42 + i)
    result = pipe(
        prompt=prompt, negative_prompt=NEGATIVE_PROMPT,
        height=IMAGE_HEIGHT, width=IMAGE_WIDTH,
        num_inference_steps=SDXL_STEPS, guidance_scale=SDXL_GUIDANCE, generator=gen,
    )
    keyframes.append(result.images[0])
    print(f"  Keyframe {i+1}/{len(shots)}: {desc[:60]}...")

# Save keyframes
kf_dir = WAREHOUSE / "outputs" / "keyframes"
kf_dir.mkdir(parents=True, exist_ok=True)
for i, kf in enumerate(keyframes):
    kf.save(kf_dir / f"kf_{i:03d}.png")
print(f"Keyframes saved to {kf_dir}")

# Unload SDXL
if has_lora:
    while hasattr(pipe.unet, "base_model"):
        try: pipe.unet = pipe.unet.base_model.model
        except: break
del pipe; gc.collect(); torch.cuda.empty_cache()
print("SDXL unloaded.")

# ================================================================
# Phase 3: AnimateDiff T2V + IP-Adapter
# ================================================================
print("\n" + "=" * 60)
print("PHASE 3: AnimateDiff T2V + IP-Adapter")
print("=" * 60)

all_clips = []
start_time = time.time()

# Load SD 1.5 LoRA path
sd15_lora_dir = None
if CHARACTER_NAME:
    sd15_dir = WAREHOUSE / "lora" / f"{char_id}_sd15"
    if (sd15_dir / "adapter_model.safetensors").exists():
        sd15_lora_dir = sd15_dir
        print(f"SD 1.5 LoRA found: {sd15_dir}")

print("\nLoading AnimateDiff pipeline...")
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler

motion_adapter = MotionAdapter.from_pretrained(
    "guoyww/animatediff-motion-adapter-v1-5-3",
    torch_dtype=torch.float16,
    cache_dir=str(WAREHOUSE / "models"),
)
print("  Motion adapter loaded")

# Load anime base model
_SD15_MODELS = ["Lykon/dreamshaper-8", "Linaqruf/anything-v3-1", "runwayml/stable-diffusion-v1-5"]
anim_pipe = None
for _model_id in _SD15_MODELS:
    try:
        anim_pipe = AnimateDiffPipeline.from_pretrained(
            _model_id, motion_adapter=motion_adapter,
            torch_dtype=torch.float16, cache_dir=str(WAREHOUSE / "models"),
        )
        print(f"  Base model: {_model_id}")
        break
    except Exception as e:
        print(f"  {_model_id} failed: {e}")

if anim_pipe is None:
    raise RuntimeError("No SD 1.5 base model available!")

anim_pipe.scheduler = DDIMScheduler.from_config(
    anim_pipe.scheduler.config, beta_schedule="linear", clip_sample=False,
)
anim_pipe.enable_vae_slicing()
anim_pipe.to("cuda")  # Direct GPU — cpu_offload hangs with IP-Adapter

# Load IP-Adapter for character conditioning
_ip_adapter_loaded = False
try:
    anim_pipe.load_ip_adapter(
        "h94/IP-Adapter", subfolder="models",
        weight_name="ip-adapter_sd15.bin",
        cache_dir=str(WAREHOUSE / "models"),
    )
    anim_pipe.set_ip_adapter_scale(IP_ADAPTER_SCALE)
    _ip_adapter_loaded = True
    print("  IP-Adapter loaded")
except Exception as e:
    print(f"  IP-Adapter not available: {e}")

# Load SD 1.5 LoRA (diffusers native — NOT PEFT, to avoid motion_module corruption)
_lora_loaded = False
if sd15_lora_dir is not None:
    try:
        import safetensors.torch
        lora_path = sd15_lora_dir / "adapter_model.safetensors"
        if lora_path.exists():
            state_dict = safetensors.torch.load_file(str(lora_path))
            # Filter out motion_module keys and strip PEFT prefix
            spatial = {}
            for k, v in state_dict.items():
                if "motion_module" in k:
                    continue
                clean_key = k.replace("base_model.model.", "")
                spatial[clean_key] = v
            if spatial:
                anim_pipe.load_lora_weights(spatial)
                _lora_loaded = True
                print(f"  Character LoRA loaded (spatial only, {len(spatial)} keys)")
    except Exception as e:
        print(f"  LoRA loading failed: {e}")

print("AnimateDiff pipeline ready!\n")

# Generate clips
ANIM_NEGATIVE = "low quality, bad anatomy, worst quality, blurry, deformed, disfigured, static, ugly, watermark, text, extra limbs"

for i, kf_image in enumerate(keyframes):
    shot = shots[i]
    char_tag = f"{CHARACTER_NAME}, " if CHARACTER_NAME else ""
    motion_prompt = f"{char_tag}anime, detailed face, expressive eyes, {shot['description']}, smooth motion, masterpiece"
    print(f"  Animating shot {i+1}/{len(keyframes)}: \"{shot['description'][:50]}...\"")

    kf_resized = kf_image.resize((512, 768), Image.LANCZOS)
    gen = torch.Generator("cpu").manual_seed(42 + i)

    try:
        gen_kwargs = dict(
            prompt=motion_prompt, negative_prompt=ANIM_NEGATIVE,
            num_frames=NUM_FRAMES, width=512, height=768,
            num_inference_steps=ANIM_STEPS, guidance_scale=ANIM_GUIDANCE,
            generator=gen,
        )
        # IP-Adapter character conditioning
        if _ip_adapter_loaded:
            gen_kwargs["ip_adapter_image"] = kf_resized

        result = anim_pipe(**gen_kwargs)
        clip_frames = result.frames[0]

        # Blend keyframe into first frames for identity anchoring
        if len(clip_frames) > 3:
            try:
                clip_w, clip_h = clip_frames[0].size if hasattr(clip_frames[0], 'size') else (np.array(clip_frames[0]).shape[1], np.array(clip_frames[0]).shape[0])
                kf_for_blend = kf_resized.resize((clip_w, clip_h), Image.LANCZOS)
                for j in range(min(3, len(clip_frames))):
                    alpha = 0.3 + (j / 3.0) * 0.7
                    frame_pil = clip_frames[j] if isinstance(clip_frames[j], Image.Image) else Image.fromarray(np.array(clip_frames[j]))
                    clip_frames[j] = Image.blend(kf_for_blend, frame_pil, alpha)
            except: pass

        all_clips.append(clip_frames)
        print(f"    {len(clip_frames)} frames generated")

    except Exception as e:
        print(f"    AnimateDiff failed: {e}")
        all_clips.append([kf_resized] * 16)
        print(f"    Using static keyframe fallback")

    gc.collect(); torch.cuda.empty_cache()

anim_time = time.time() - start_time
print(f"\nAnimateDiff done in {anim_time/60:.1f} min")

# Cleanup
if _lora_loaded:
    try: anim_pipe.unload_lora_weights()
    except: pass
del anim_pipe, motion_adapter
gc.collect(); torch.cuda.empty_cache()

# ================================================================
# Phase 4: Post-Processing
# ================================================================
print("\n" + "=" * 60)
print("PHASE 4: Post-Processing")
print("=" * 60)

# 4a. Trim static tails
try:
    from agents.postprocess.motion_trim import MotionTrimmer
    trimmer = MotionTrimmer()
    for idx, clip in enumerate(all_clips):
        orig = len(clip)
        all_clips[idx] = trimmer.trim_static_tail(clip)
        trimmed = orig - len(all_clips[idx])
        if trimmed > 0:
            print(f"  Clip {idx+1}: trimmed {trimmed} static frames ({orig} -> {len(all_clips[idx])})")
except Exception as e:
    print(f"  Motion trim skipped: {e}")

# 4b. Face restoration
if FACE_RESTORE:
    try:
        from agents.postprocess.face_restore import AnimeFaceRestorer
        restorer = AnimeFaceRestorer()
        print("Restoring anime faces...")
        for idx, clip in enumerate(all_clips):
            all_clips[idx] = restorer.restore_frames(clip)
        print("  Done")
    except Exception as e:
        print(f"  Face restore skipped: {e}")

# 4c. Spatial upscale / sharpening
if SPATIAL_UPSCALE:
    try:
        from basicsr.archs.rrdbnet_arch import RRDBNet
        from realesrgan import RealESRGANer
        esrgan_model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=6, num_grow_ch=32, scale=4)
        esrgan = RealESRGANer(
            scale=4,
            model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth",
            model=esrgan_model, half=True,
        )
        print("Sharpening with Real-ESRGAN...")
        for idx, clip in enumerate(all_clips):
            sharpened = []
            for frame_pil in clip:
                frame_bgr = cv2.cvtColor(np.array(frame_pil), cv2.COLOR_RGB2BGR)
                h_orig, w_orig = frame_bgr.shape[:2]
                out_bgr, _ = esrgan.enhance(frame_bgr, outscale=1)
                if out_bgr.shape[:2] != (h_orig, w_orig):
                    out_bgr = cv2.resize(out_bgr, (w_orig, h_orig), interpolation=cv2.INTER_LANCZOS4)
                sharpened.append(Image.fromarray(cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)))
            all_clips[idx] = sharpened
        del esrgan, esrgan_model; gc.collect(); torch.cuda.empty_cache()
        print("  Done")
    except Exception as e:
        print(f"  Real-ESRGAN skipped: {e}")

# 4d. Color grading
if COLOR_GRADE:
    try:
        from agents.postprocess.color_grade import AnimeColorGrader
        grader = AnimeColorGrader()
        print(f"Color grading ({COLOR_PALETTE})...")
        for idx, clip in enumerate(all_clips):
            all_clips[idx] = grader.grade_with_palette(clip, palette=COLOR_PALETTE)
        print("  Done")
    except Exception as e:
        print(f"  Color grading skipped: {e}")

# ================================================================
# Phase 5: Temporal Upscale (RIFE 8fps -> 24fps)
# ================================================================
if TARGET_FPS > FPS:
    print(f"\nTemporal upscale: {FPS}fps -> {TARGET_FPS}fps")
    multiplier = round(TARGET_FPS / FPS)
    rife_ok = False
    try:
        from agents.postprocess.upscaler import VideoUpscaler
        _up = VideoUpscaler(str(WAREHOUSE))
        _rife = _up._load_rife()
        if _rife: rife_ok = True; print("  Using RIFE (optical flow)")
    except: pass

    for idx, clip in enumerate(all_clips):
        orig = len(clip)
        if rife_ok:
            try:
                all_clips[idx] = _up._rife_interpolate_sequence(_rife, clip, multiplier)
            except:
                upscaled = []
                for j in range(len(clip) - 1):
                    upscaled.append(clip[j])
                    a1, a2 = np.array(clip[j]).astype(np.float32), np.array(clip[j+1]).astype(np.float32)
                    for k in range(1, multiplier):
                        alpha = k / multiplier
                        upscaled.append(Image.fromarray(((1-alpha)*a1 + alpha*a2).astype(np.uint8)))
                upscaled.append(clip[-1])
                all_clips[idx] = upscaled
        else:
            upscaled = []
            for j in range(len(clip) - 1):
                upscaled.append(clip[j])
                a1, a2 = np.array(clip[j]).astype(np.float32), np.array(clip[j+1]).astype(np.float32)
                for k in range(1, multiplier):
                    alpha = k / multiplier
                    upscaled.append(Image.fromarray(((1-alpha)*a1 + alpha*a2).astype(np.uint8)))
            upscaled.append(clip[-1])
            all_clips[idx] = upscaled
        print(f"  Clip {idx+1}: {orig} -> {len(all_clips[idx])} frames")

    if rife_ok:
        try: _up._unload_rife()
        except: pass
        gc.collect(); torch.cuda.empty_cache()
    output_fps = TARGET_FPS
else:
    output_fps = FPS

# ================================================================
# Phase 6: Assembly — cross-dissolve stitch
# ================================================================
print(f"\nAssembling {len(all_clips)} clips...")
CROSSFADE = 6
all_frames = []
for idx, clip in enumerate(all_clips):
    arrays = [np.array(f) for f in clip]
    if idx == 0:
        all_frames.extend(arrays)
    else:
        overlap = min(CROSSFADE, len(all_frames), len(arrays))
        for j in range(overlap):
            t = (j + 1) / (overlap + 1)
            prev = all_frames[-(overlap - j)].astype(np.float32)
            nxt = arrays[j].astype(np.float32)
            all_frames[-(overlap - j)] = ((1-t)*prev + t*nxt).clip(0, 255).astype(np.uint8)
        all_frames.extend(arrays[overlap:])

duration = len(all_frames) / output_fps
print(f"Total: {len(all_frames)} frames ({duration:.1f}s at {output_fps}fps)")

# Save video
output_dir = WAREHOUSE / "outputs" / "text2anime"
output_dir.mkdir(parents=True, exist_ok=True)
video_path = output_dir / f"anime_{int(time.time())}.mp4"

h, w = all_frames[0].shape[:2]
writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), output_fps, (w, h))
for frame in all_frames:
    writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
writer.release()

total_time = time.time() - start_time
size_mb = video_path.stat().st_size / 1e6

print(f"\n{'=' * 60}")
print(f"DONE!")
print(f"{'=' * 60}")
print(f"Video: {video_path}")
print(f"Duration: {duration:.1f}s | Resolution: {w}x{h} | FPS: {output_fps}")
print(f"Size: {size_mb:.1f}MB | Time: {total_time/60:.1f} min")

# Show keyframes preview
import matplotlib.pyplot as plt
cols = min(len(keyframes), 4)
rows = (len(keyframes) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
axes_flat = np.array(axes).flat if hasattr(axes, 'flat') else [axes]
for i, (ax, kf) in enumerate(zip(axes_flat, keyframes)):
    ax.imshow(kf)
    ax.set_title(f"Shot {i+1}: {shots[i]['description'][:35]}...", fontsize=9)
    ax.axis("off")
for j in range(i+1, rows*cols):
    try: axes_flat[j].axis("off")
    except: pass
plt.suptitle("Generated Keyframes", fontsize=13)
plt.tight_layout()
plt.show()

gc.collect(); torch.cuda.empty_cache()
print("VRAM freed.")

In [ ]:
#@title 4. Play Video
#@markdown Displays the generated video inline.

from IPython.display import Video, display
from pathlib import Path
import glob

# Find latest video
output_dir = Path("/workspace/warehouse/outputs/text2anime")
videos = sorted(output_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime, reverse=True)
if videos:
    latest = videos[0]
    print(f"Playing: {latest}")
    print(f"Size: {latest.stat().st_size / 1e6:.1f}MB")
    display(Video(str(latest), embed=True, width=512))
else:
    print("No videos found. Run Cell 3 first.")

In [ ]:
#@title 5. Download Video
#@markdown Downloads the latest video to your local machine.

from pathlib import Path
from IPython.display import FileLink

output_dir = Path("/workspace/warehouse/outputs/text2anime")
videos = sorted(output_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime, reverse=True)
if videos:
    print(f"Right-click to download: {videos[0].name}")
    display(FileLink(str(videos[0])))
else:
    print("No videos found.")

In [ ]:
#@title 6. Stop Pod (saves money!)
#@markdown Stops the pod to avoid charges. Your /workspace volume persists.

STOP_POD = False  #@param {type:"boolean"}

if STOP_POD:
    import subprocess
    subprocess.run(["runpodctl", "stop", "pod", "$RUNPOD_POD_ID"], check=False)
    print("Pod stopping... Your volume data is preserved.")
else:
    print("Toggle STOP_POD to True and re-run to stop.")